# Validación del Pipeline de Carga — MODO DESARROLLO
RAG Inmobiliario Montevideo

Este notebook cubre el flujo completo previo a indexar:

1. **Setup** — directorio de trabajo y autoreload
2. **Segmentos disponibles** — verificar cobertura del CSV
3. **Spot check por segmento** — revisar el `page_content` generado
4. **Validaciones estructurales** — documentos vacíos, metadata keys
5. **Análisis de tokens** — verificar que ningún documento supera el límite de `gemini-embedding-001` (2048 tokens)
6. **Construcción del índice FAISS**
7. **Validación del índice** — búsqueda semántica y búsqueda con filtros

## 1. Setup

In [1]:
import os
import sys
import importlib

%load_ext autoreload
%autoreload 2

ROOT = "/Users/alejandra/DataAnalytics-UniAndes/Proyecto/realstate_ragas"  # <- ajustar
sys.path.insert(0, ROOT)
os.chdir(ROOT)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/alejandra/DataAnalytics-UniAndes/Proyecto/realstate_ragas


In [2]:
import numpy as np
import pandas as pd

from app.services.csv_document_service import (
    CSVDocumentService,
    preprocess_dataframe,
    _build_page_content,
)

CSV_PATH = "./docs/realstate_mvd/listings.csv"

svc = CSVDocumentService()
print("CSVDocumentService OK")

CSVDocumentService OK


## 2. Segmentos disponibles

In [5]:
segments = svc.get_available_segments(CSV_PATH)
print(segments)
# Esperado: ~3.377 listings, 4 combinaciones de tipo de operación/propiedad, ~50 barrios

# Previsualizar los primeros 3 documentos
svc.preview_document("./docs/realstate_mvd/listings.csv", n=3)


{'operation_types': ['alquiler', 'venta'], 'property_types': ['apartamentos', 'casas'], 'barrios': ['AGUADA', 'AIRES PUROS', 'ATAHUALPA', 'BARRIO SUR', 'BAÑADOS DE CARRASCO', 'BELVEDERE', 'BRAZO ORIENTAL', 'BUCEO', 'CAPURRO BELLA VISTA', 'CARRASCO', 'CARRASCO NORTE', 'CASABO PAJAS BLANCAS', 'CASAVALLE', 'CASTRO CASTELLANOS', 'CENTRO', 'CERRITO', 'CERRO', 'CIUDAD VIEJA', 'COLON CENTRO Y NOROESTE', 'COLON SURESTE ABAYUBA', 'CONCILIACION', 'CORDON', 'FIGURITA', 'FLOR DE MAROÑAS', 'ITUZAINGO', 'JACINTO VERA', 'JARDINES DEL HIPODROMO', 'LA BLANQUEADA', 'LA COMERCIAL', 'LA TEJA', 'LARRAÑAGA', 'LAS ACACIAS', 'LAS CANTERAS', 'LEZICA MELILLA', 'MALVIN', 'MALVIN NORTE', 'MANGA', 'MANGA TOLEDO CHICO', 'MAROÑAS PARQUE GUARANI', 'MERCADO MODELO Y BOLIVAR', 'NUEVO PARIS', 'PALERMO', 'PARQUE RODO', 'PASO DE LA ARENA', 'PASO DE LAS DURANAS', 'PEÑAROL LAVALLEJA', 'PIEDRAS BLANCAS', 'POCITOS', 'PQUE BATLLE VILLA DOLORES', 'PRADO NUEVA SAVONA', 'PUNTA CARRETAS', 'PUNTA GORDA', 'PUNTA RIELES BELLA ITALIA'

In [6]:
# Informacion general listings:
print(f"Total listings      : {segments['total_listings']}")
print(f"Tipos de operación  : {segments['operation_types']}")
print(f"Tipos de propiedad  : {segments['property_types']}")
print(f"Barrios             : {len(segments['barrios'])} únicos")
print()
print("Barrios:")
for b in segments['barrios']:
    print(f"  {b}")

Total listings      : 3377
Tipos de operación  : ['alquiler', 'venta']
Tipos de propiedad  : ['apartamentos', 'casas']
Barrios             : 61 únicos

Barrios:
  AGUADA
  AIRES PUROS
  ATAHUALPA
  BARRIO SUR
  BAÑADOS DE CARRASCO
  BELVEDERE
  BRAZO ORIENTAL
  BUCEO
  CAPURRO BELLA VISTA
  CARRASCO
  CARRASCO NORTE
  CASABO PAJAS BLANCAS
  CASAVALLE
  CASTRO CASTELLANOS
  CENTRO
  CERRITO
  CERRO
  CIUDAD VIEJA
  COLON CENTRO Y NOROESTE
  COLON SURESTE ABAYUBA
  CONCILIACION
  CORDON
  FIGURITA
  FLOR DE MAROÑAS
  ITUZAINGO
  JACINTO VERA
  JARDINES DEL HIPODROMO
  LA BLANQUEADA
  LA COMERCIAL
  LA TEJA
  LARRAÑAGA
  LAS ACACIAS
  LAS CANTERAS
  LEZICA MELILLA
  MALVIN
  MALVIN NORTE
  MANGA
  MANGA TOLEDO CHICO
  MAROÑAS PARQUE GUARANI
  MERCADO MODELO Y BOLIVAR
  NUEVO PARIS
  PALERMO
  PARQUE RODO
  PASO DE LA ARENA
  PASO DE LAS DURANAS
  PEÑAROL LAVALLEJA
  PIEDRAS BLANCAS
  POCITOS
  PQUE BATLLE VILLA DOLORES
  PRADO NUEVA SAVONA
  PUNTA CARRETAS
  PUNTA GORDA
  PUNTA RIELES BEL

## 3. Spot check por segmento

Un listing por cada combinación operación × tipo de propiedad. Verificar:
- `Ubicación:` aparece cuando hay datos de dirección (`l3`)
- `planta baja` para piso 0
- `a estrenar` para `age=0` o `condition=new`
- Máximo 3–4 cocheras
- Amenities (`piscina`, `terraza`, `parrillero`) donde corresponde
- `Disponible para venta y alquiler.` en listings `is_dual_intent=True`
- Listings con `barrio_confidence='marketing_inflation'` usan `title_clean` / `desc_clean`
- Descripciones largas terminan en `...` (truncadas a 5.000 chars)

In [7]:
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df = preprocess_dataframe(df_raw)

print(f"Filas: {len(df)}  |  Columnas: {df.shape[1]}")

[preprocess] Flags desde amenities:   {'has_elevator': 1224, 'has_gym': 448, 'has_rooftop': 219, 'has_party_room': 259, 'has_multipurpose_room': 605, 'has_laundry': 409, 'has_green_area': 447, 'has_cowork': 182, 'has_internet': 635, 'has_wheelchair': 260, 'has_fireplace': 262, 'has_fridge': 173, 'has_parrillero': 202, 'has_reception': 287, 'has_playground': 195, 'has_visitor_parking': 132, 'has_sauna': 55}
[preprocess] Columnas eliminadas: 26
[preprocess] Columnas restantes:  76
[preprocess] Flags desde description: {'has_pool': 414, 'has_parrillero': 1195, 'has_terrace': 1440, 'has_storage': 233, 'has_security': 794}
Filas: 3377  |  Columnas: 80


In [8]:
# Un listing por segmento (posición 10 para evitar casos borde al inicio)
for op in ["venta", "alquiler"]:
    for pt in ["apartamentos", "casas"]:
        subset = df[(df["operation_type"] == op) & (df["property_type"] == pt)]
        if len(subset) == 0:
            print(f"[WARNING] Sin datos para {op} / {pt}")
            continue
        idx = min(10, len(subset) - 1)
        row = subset.iloc[idx]
        print(f"\n{'='*60}")
        print(f"SEGMENTO: {op} / {pt}  (barrio_confidence={row.get('barrio_confidence', '?')})")
        print("="*60)
        print(_build_page_content(row))


SEGMENTO: venta / apartamentos  (barrio_confidence=no_barrio_in_text)
Apartamento en venta en COLON CENTRO Y NOROESTE.
3 dormitorios, 1 baño, piso 1, a estrenar.
Superficie cubierta 50 m², total 55 m².
Precio: USD 30.000 (600 USD/m²).
Ubicación: Carlos A. Lopez 8440, Colón
Amenities: terraza.
Entorno: 7 espacios verdes en radio de 800 m, 3 escuelas en radio de 800 m (la más cercana a 180 m), 2 primarias en zona, zona comercial con múltiples servicios, zona con 3 establecimientos industriales en radio de 800 m.
Título: Vendo: Apartamento En Complejo Artigas De 3 Dormitorios
Descripción: INMOBILIARIA ARTEAGA HILL COLON VENDE: Apartamento en Complejo Artigas, sobre calle Carlos A. Lopez, a pocos metros de Cesar Mayo Gutierrez. 
Primer piso por escalera.

Living comedor amplio, con cocina semi-integrada.
Terraza lavadero.
3 dormitorios.
Baño.
Cochera de uso exclusivo.

Bajos gastos comunes, $ 1900 aprox.

El precio publicado es la entrega, tiene saldo a pagar en la ANV de 390 UR (usd 18.5

In [9]:
# Spot check específico: listings dual intent
dual = df[df["is_dual_intent"] == True]
print(f"Listings dual intent: {len(dual)}")
if len(dual) > 0:
    print()
    print(_build_page_content(dual.iloc[0]))

Listings dual intent: 90

Apartamento en alquiler en PUNTA CARRETAS.
3 dormitorios, 4 baños, 2 cocheras, piso 7, 29 años de antigüedad.
Superficie cubierta 126 m², total 141 m².
Precio: UYU 93.000 (738 USD/m²).
Expensas: UYU 29.000.
Ubicación: Terú y Rambla Gandhi, Punta Carretas
Amenities: ascensor, parrillero, terraza.
Entorno: a 445 m de la playa, plaza a 215 m, 3 plazas en radio de 800 m, 3 escuelas en radio de 800 m (la más cercana a 600 m), 2 primarias, 1 liceo en zona, zona comercial con múltiples servicios.
Disponible para venta y alquiler.
Título: Venta Y Alquiler Apartamento Punta Carretas 3 Dormitorios Teru Y Rambla Vista Mar 2 Garajes Parrilla
Descripción: - INMOBILIARIA MONYMAR -

TERÚ Y RAMBLA GANDHI... CORAZON DE VILLA BIARRITZ Y PUNTA CARRETAS CON VISTA AL MAR Y AL PARQUE... IMPECABLE ESTADO... AL FRENTE... Living comedor con salida a TERRAZA con VISTA AL MAR, 3 DORMITORIOS (uno en suite) y salida a TERRAZA, 4 baños (3 completos y uno social), SERVICIO COMPLETO. ESTAR D

In [10]:
# Spot check específico: listings marketing_inflation (deben usar title_clean / desc_clean)
inflation = df[df["barrio_confidence"] == "marketing_inflation"]
print(f"Listings marketing_inflation: {len(inflation)}")
if len(inflation) > 0:
    row = inflation.iloc[0]
    print(f"\ntitle original : {row.get('title', '')}")
    print(f"title_clean    : {row.get('title_clean', '')}")
    print()
    print(_build_page_content(row))

Listings marketing_inflation: 85

title original : Departamento En  Proa Carrasco - Unidad 406 - Penthouse Amoblado - 101,85 M2 - 2 Dorm - 2 Baños - 1 Cochera - 1 Baulera
title_clean    : Departamento En  Proa Carrasco Norte - Unidad 406 - Penthouse Amoblado - 101,85 M2 - 2 Dorm - 2 Baños - 1 Cochera - 1 Baulera

Apartamento en alquiler en CARRASCO NORTE.
2 dormitorios, 2 baños, 1 cochera, 2 años de antigüedad.
Superficie cubierta 101 m².
Precio: UYU 66.960 (662 USD/m²).
Expensas: UYU 16.000.
Ubicación: Proa Carrasco - Unidad 406 - Penthouse Amoblado - 101,85 m2 - 2 Dorm - 2 Baños - 1 Cochera - 1 Baulera, Carrasco
Amenities: ascensor, gimnasio, salón de usos múltiples, área de lavandería, área verde, acceso a internet, parrillero, área de juegos infantiles, piscina, terraza, depósito/baulera, seguridad/vigilancia.
Entorno: plaza a 199 m, 2 escuelas en radio de 800 m (la más cercana a 374 m), 1 primaria en zona, zona comercial con múltiples servicios, zona con 1 establecimiento industri

## 4. Validaciones estructurales

In [11]:
docs = svc.dataframe_to_documents(df)

[documents] Creados: 3377


In [12]:
# Documentos vacíos
empty = [i for i, d in enumerate(docs) if not d.page_content.strip()]
print(f"Documentos vacíos: {len(empty)}")
if empty:
    print(f"  Índices: {empty[:10]}")

Documentos vacíos: 0


In [13]:
# Metadata keys obligatorias
expected_keys = {
    "id", "operation_type", "property_type", "barrio_fixed",
    "price_fixed", "barrio_confidence", "is_dual_intent", "source"
}

missing_report = []
for i, doc in enumerate(docs):
    missing = expected_keys - set(doc.metadata.keys())
    if missing:
        missing_report.append((i, missing))

if missing_report:
    print(f"[WARNING] {len(missing_report)} docs con keys faltantes:")
    for idx, keys in missing_report[:5]:
        print(f"  Doc {idx}: {keys}")
else:
    print("Metadata keys: OK — todos los documentos tienen las keys esperadas")

Metadata keys: OK — todos los documentos tienen las keys esperadas


In [14]:
# Distribución de barrio_confidence
confidence_vals = [d.metadata.get("barrio_confidence", None) for d in docs]
print("Distribución barrio_confidence:")
print(pd.Series(confidence_vals).value_counts().to_string())

Distribución barrio_confidence:
consistent             1858
no_barrio_in_text       816
genuine_ambiguity       618
marketing_inflation      85


## 5. Análisis de tokens

Límite del modelo: **2048 tokens** (`gemini-embedding-001`).  
Se mide con el tokenizador real sobre una muestra de 200 documentos y se extrapola con la aproximación `chars / 3.2` al corpus completo.

In [19]:
import google.generativeai as genai

# Verificar límites del modelo
model_info = genai.get_model("models/gemini-embedding-001")
print(f"Input token limit : {model_info.input_token_limit}")
print(f"Version           : {model_info.version}")

# Embedding dimension:
test_embedding = genai.embed_content(
    model="models/gemini-embedding-001",
    content="test"
)
print(f"Embedding dimension: {len(test_embedding['embedding'])}")

Input token limit : 2048
Output token limit: 1
Version           : 001
Embedding dimension: 3072


In [21]:
print(model_info.supported_generation_methods)

['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']


In [22]:
# Medición exacta en muestra de 200 docs (evita costos innecesarios)
import random

TOKEN_LIMIT = 2048
SAMPLE_SIZE = 200

random.seed(42)
sample_docs = random.sample(docs, min(SAMPLE_SIZE, len(docs)))

gemini_model = genai.GenerativeModel("models/gemini-embedding-001")
sample_tokens = np.array([
    gemini_model.count_tokens(d.page_content).total_tokens
    for d in sample_docs
])

char_lengths = np.array([len(d.page_content) for d in sample_docs])
ratios       = char_lengths / sample_tokens

print("── Distribución de tokens (muestra 200 docs) ──────────────────")
print(pd.Series(sample_tokens).describe(percentiles=[.50, .75, .90, .95, .99]).to_string())
print(f"\nDocs sobre límite ({TOKEN_LIMIT}): {(sample_tokens > TOKEN_LIMIT).sum()} / {len(sample_tokens)}")
print(f"\nRatio chars/token  media={ratios.mean():.2f}  "
      f"p5={np.percentile(ratios,5):.2f}  p95={np.percentile(ratios,95):.2f}")

── Distribución de tokens (muestra 200 docs) ──────────────────
count    200.000000
mean     435.275000
std      124.031482
min      161.000000
50%      426.500000
75%      498.250000
90%      588.200000
95%      642.450000
99%      847.090000
max      913.000000

Docs sobre límite (2048): 0 / 200

Ratio chars/token  media=3.52  p5=2.95  p95=4.11


In [23]:
# Aproximación chars/3.2 sobre el corpus completo
# (conservadora: sobreestima ligeramente los tokens)
approx_tokens = np.array([len(d.page_content) / 3.2 for d in docs])

# Separar porción no-descripción vs descripción
non_desc_tokens = np.array([
    len(d.page_content.split("Descripción:")[0]) / 3.2
    for d in docs
])
desc_tokens = approx_tokens - non_desc_tokens

print("── Distribución aprox. tokens — corpus completo (chars/3.2) ───")
print(pd.Series(approx_tokens).describe(percentiles=[.50, .75, .90, .95, .99]).to_string())
print(f"\nDocs sobre límite ({TOKEN_LIMIT}): {(approx_tokens > TOKEN_LIMIT).sum()} / {len(approx_tokens)} "
      f"({(approx_tokens > TOKEN_LIMIT).mean()*100:.2f}%)")

print(f"\n── Tokens sin descripción (p50 / p95 / max) ───────────────────")
print(f"  p50={np.percentile(non_desc_tokens,50):.0f}  "
      f"p95={np.percentile(non_desc_tokens,95):.0f}  "
      f"max={non_desc_tokens.max():.0f}")

print(f"\n── Tokens de descripción (p50 / p95 / max) ────────────────────")
print(f"  p50={np.percentile(desc_tokens,50):.0f}  "
      f"p95={np.percentile(desc_tokens,95):.0f}  "
      f"max={desc_tokens.max():.0f}")

truncated = sum(
    1 for d in docs
    if d.page_content.endswith("...")
)
print(f"\nDescripciones truncadas (terminan en '...'): {truncated}")

── Distribución aprox. tokens — corpus completo (chars/3.2) ───
count    3377.000000
mean      479.518526
std       173.571214
min       102.500000
50%       450.625000
75%       571.875000
90%       697.312500
95%       795.375000
99%      1012.112500
max      1614.687500

Docs sobre límite (2048): 0 / 3377 (0.00%)

── Tokens sin descripción (p50 / p95 / max) ───────────────────
  p50=170  p95=218  max=286

── Tokens de descripción (p50 / p95 / max) ────────────────────
  p50=275  p95=623  max=1352

Descripciones truncadas (terminan en '...'): 6


## 6. Construcción del índice FAISS

Este paso realiza llamadas a la API de embeddings (~3–4 min, ~$0.03).  
Ejecutar solo después de que las validaciones anteriores sean satisfactorias.

Si el proceso se interrumpe, simplemente volver a ejecutar la celda — retomará desde el último checkpoint.

In [25]:
from app.services.load_documents_service import load_from_csv

result = await load_from_csv("realstate_mvd")

print(f"Éxito               : {result['success']}")
print(f"Mensaje             : {result['message']}")
print(f"Documentos indexados: {result['data']['processing_summary']['total_documents']}")
print(f"Costo estimado      : ${result['data']['embedding_statistics']['cost_breakdown']['embedding_cost_usd']}")


[ChunkingService] Procesando CSV: docs/realstate_mvd/listings.csv

CARGA DESDE CSV: docs/realstate_mvd/listings.csv
Filas cargadas: 3377
[preprocess] Flags desde amenities:   {'has_elevator': 1224, 'has_gym': 448, 'has_rooftop': 219, 'has_party_room': 259, 'has_multipurpose_room': 605, 'has_laundry': 409, 'has_green_area': 447, 'has_cowork': 182, 'has_internet': 635, 'has_wheelchair': 260, 'has_fireplace': 262, 'has_fridge': 173, 'has_parrillero': 202, 'has_reception': 287, 'has_playground': 195, 'has_visitor_parking': 132, 'has_sauna': 55}
[preprocess] Columnas eliminadas: 26
[preprocess] Columnas restantes:  76
[preprocess] Flags desde description: {'has_pool': 414, 'has_parrillero': 1195, 'has_terrace': 1440, 'has_storage': 233, 'has_security': 794}


CLOUD_STORAGE_BUCKET no configurada. Los índices se guardarán localmente.


[documents] Creados: 3377

[ChunkingService] 3377 documents listos (colección: realstate_mvd)

CONSTRUCCIÓN DE VECTORSTORE FAISS
Total chunks: 3377
Modelo: models/gemini-embedding-001


GENERACIÓN DE EMBEDDINGS
Total textos:    3377
Batch size:      50
Total batches:   68
Delay entre batches: 15s

[INFO] Procesando batch 1/68 (50 textos)...
[INFO] ✓ Batch 1 completado en 3.06s
[INFO] Esperando 15s...
[INFO] Procesando batch 2/68 (50 textos)...
[INFO] ✓ Batch 2 completado en 3.02s
[INFO] Esperando 15s...
[INFO] Procesando batch 3/68 (50 textos)...
[INFO] ✓ Batch 3 completado en 3.00s
[INFO] Esperando 15s...
[INFO] Procesando batch 4/68 (50 textos)...
[INFO] ✓ Batch 4 completado en 3.07s
[INFO] Esperando 15s...
[INFO] Procesando batch 5/68 (50 textos)...
[INFO] ✓ Batch 5 completado en 2.59s
[INFO] Esperando 15s...
[INFO] Procesando batch 6/68 (50 textos)...
[INFO] ✓ Batch 6 completado en 2.65s
[INFO] Esperando 15s...
[INFO] Procesando batch 7/68 (50 textos)...
[INFO] ✓ Batch 7 completado

## 7. Validación del índice

Verificar que el índice responde correctamente a búsqueda semántica libre y a búsqueda filtrada.

In [26]:
from app.services.embedding_service import EmbeddingService
from app.services.retrieval_service import RetrievalService, PropertyFilters

emb = EmbeddingService()
emb.load_vectorstore("./faiss_index/realstate_mvd")
ret = RetrievalService(emb, k=5)

print("Índice cargado OK")

CLOUD_STORAGE_BUCKET no configurada. Los índices se guardarán localmente.


Índice cargado OK


In [27]:
# Test 1 — búsqueda semántica sin filtros
query = "apartamento con piscina cerca del mar"
results = ret.retrieve_documents(query)

print(f"Query: '{query}'")
print(f"Resultados: {len(results)}")
print()
for i, d in enumerate(results):
    print(f"── Resultado {i+1} ──────────────────────────────────")
    print(d.page_content[:200])
    print()

Query: 'apartamento con piscina cerca del mar'
Resultados: 5

── Resultado 1 ──────────────────────────────────
Apartamento en venta en POCITOS.
1 dormitorio, 1 baño, piso 3, a estrenar.
Superficie cubierta 40 m², total 48 m².
Precio: USD 178.000 (4.450 USD/m²).
Expensas: UYU 6.000.
Ubicación: 26 de Marzo 1263,

── Resultado 2 ──────────────────────────────────
Apartamento en venta en POCITOS.
1 dormitorio, 1 baño, piso 3, a estrenar.
Superficie cubierta 40 m², total 48 m².
Precio: USD 185.000 (4.625 USD/m²).
Expensas: UYU 6.000.
Ubicación: 26 de Marzo 1263,

── Resultado 3 ──────────────────────────────────
Apartamento en alquiler en POCITOS.
1 dormitorio, 1 baño, 1 cochera, piso 2, 10 años de antigüedad.
Superficie cubierta 47 m².
Precio: UYU 32.000 (680 USD/m²).
Expensas: UYU 8.000.
Ubicación: Sin dire

── Resultado 4 ──────────────────────────────────
Apartamento en alquiler en POCITOS.
2 dormitorios, 2 baños, piso 12, 2 años de antigüedad.
Superficie cubierta 65 m².
Precio: UYU 45

In [28]:
# Test 2 — búsqueda con filtros
filters = PropertyFilters(
    operation_type="venta",
    barrio="POCITOS",
    max_price=200_000
)
results = ret.retrieve_with_filters("apartamento luminoso", filters)

print(f"Resultados con filtros (venta / POCITOS / max $200k): {len(results)}")
if len(results) == 0:
    print("[HINT] Si retorna 0, aumentar fetch_k: RetrievalService(emb, k=5, fetch_k=200)")
print()
for i, d in enumerate(results):
    relevant = {k: v for k, v in d.metadata.items()
                if k in ["barrio_fixed", "operation_type", "price_fixed",
                         "barrio_confidence", "is_dual_intent"]}
    print(f"── Resultado {i+1}: {relevant}")

[RetrievalService] retrieve_with_filters: 1 docs retrieved (operation=venta, property=None, barrio=POCITOS)
Resultados con filtros (venta / POCITOS / max $200k): 1

── Resultado 1: {'operation_type': 'venta', 'barrio_fixed': 'POCITOS', 'barrio_confidence': 'consistent', 'is_dual_intent': False, 'price_fixed': 115000.0}


In [29]:
# Test 3 — verificar que barrio_confidence y is_dual_intent están presentes en metadata
print("Sample de metadata (primer resultado):")
if results:
    for k, v in results[0].metadata.items():
        if v is not None:
            print(f"  {k}: {v}")

Sample de metadata (primer resultado):
  id: MLU1041472018
  operation_type: venta
  property_type: apartamentos
  barrio_fixed: POCITOS
  barrio_confidence: consistent
  is_dual_intent: False
  lat: -34.905039
  lon: -56.1446641
  price_fixed: 115000.0
  currency_fixed: USD
  price_m2: 3833.3333333333335
  price_m2_basis: covered
  surface_covered: 30.0
  surface_total: 30.0
  bedrooms: 0.0
  bathrooms: 1.0
  floor: 3.0
  age: 2.0
  condition: used
  garages: 0.0
  expenses: 2400.0
  has_elevator: 1
  has_gym: 0
  has_rooftop: 0
  has_party_room: 0
  has_multipurpose_room: 1
  has_laundry: 0
  has_green_area: 0
  has_cowork: 1
  has_internet: 0
  has_wheelchair: 0
  has_fireplace: 0
  has_fridge: 0
  has_parrillero: 0
  has_reception: 0
  has_playground: 0
  has_visitor_parking: 0
  has_sauna: 0
  has_pool: 0
  has_terrace: 0
  has_storage: 0
  has_security: 1
  dist_nearest_public_space: 189.5108627975945
  dist_espacio_libre: 261.4724995617547
  dist_plaza: 272.76320985405096
  dist